In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import (KFold, GroupKFold, RandomizedSearchCV, GroupShuffleSplit, ShuffleSplit,
                                     cross_validate, cross_val_predict)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor

from scipy.stats import spearmanr
from sklearn.inspection import permutation_importance
import shap


# # ----------------------------------------------------------------------------
# # 0. SETUP
# # ----------------------------------------------------------------------------
TARGET = "prop_floodable_area_flooded" # "10cm_area"
GROUP_COL = None          # e.g. "catchment_id" / "cell_id" -- STRONGLY recommended if spatial/temporal autocorrelation
# LOG_TARGET = False        # set True if the target is right-skewed / spans orders of magnitude
SEED = 42
# N_OUTER, N_INNER, N_ITER = 5, 3, 40
# N_JOBS = -1

from PrepDataFunctions import *

### Prepare data

In [14]:
df = prep_for_modelling(f"/scratch/hydro4/users/kv25483/FutureFlood/Data/EventDetails_v4/all_catchments_with_topo_and_temporal.pkl", 
                        remove_events_with_zero_flooding = False)
df = df[df['mean_sm_2_before_event'].notna()]

cols = ["x_coord", "y_coord"]
df["peak_cell_id"] = df.groupby(cols, sort=False).ngroup()

subset_early = create_temporal_subset(df, "before", 2020).copy()
print(len(subset_early))

23526


In [17]:
subset_early = subset_early[subset_early['modality'] =='unimodal']
print(len(subset_early))
ev = subset_early

20923


In [15]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
ev=subset_early

# --- 2. split events by how much rain falls outside the peak step ---
ev["peak_class"] = pd.cut(ev["peak_share"], [0, 0.5, 0.8, 1.0], labels=["spread", "mixed", "peak-dominated"], include_lowest=True)
print(ev["peak_class"].value_counts())
 
# --- 3. does the temporal information help, and only in some classes? (cross-validated R2 with vs without) ---
base = ["max_precip", "total_acc"]
temporal = ["rainfall_before_peak", "loading_index"]            # or ["d50_n"]; compare the options
y = ev["10cm_area"]
 
def cv_r2(d, cols, target="10cm_area"):
    p = cross_val_predict(LinearRegression(), d[cols], d[target], cv=KFold(5, shuffle=True, random_state=0))
    return r2_score(d[target], p)
 
rows = []
for cls, g in ev.groupby("peak_class", observed=True):
    if len(g) < 100: continue
    rows.append({"class": cls, "n": len(g), "R2_base": cv_r2(g, base), "R2_with_temporal": cv_r2(g, base + temporal),
                 "R2_with_d50": cv_r2(g, base + ["d50_n"])})
print(pd.DataFrame(rows).round(3).to_string(index=False))
 
# --- 4. or one model with an interaction, so the effect scales with how much rain is outside the peak ---
ev["d50_x_nonpeak"] = ev["d50_n"] * ev["nonpeak_share"]
print("CV R2, with interaction:", round(cv_r2(ev, base + ["nonpeak_share", "d50_n", "d50_x_nonpeak"]), 3))
 

peak_class
mixed             11629
peak-dominated     6626
spread             2668
Name: count, dtype: int64
         class     n  R2_base  R2_with_temporal  R2_with_d50
        spread  2668    0.054             0.062        0.060
         mixed 11629    0.151             0.155        0.154
peak-dominated  6626    0.255             0.256        0.255
CV R2, with interaction: 0.167


In [16]:
cols = ["max_precip", "total_acc", "d50_n", "nonpeak_share", "10cm_area"]
within = ev[cols] - ev.groupby("peak_cell_id")[cols].transform("mean")        # event-to-event variation within each pixel
within["peak_class"] = ev["peak_class"]
print("pooled base:", round(cv_r2(ev, ["max_precip", "total_acc"]), 3))
print("within-pixel, rainfall only:", round(cv_r2(within, ["max_precip", "total_acc"]), 3))
print("within-pixel, + temporal:", round(cv_r2(within, ["max_precip", "total_acc", "d50_n", "nonpeak_share"]), 3))

pooled base: 0.162
within-pixel, rainfall only: 0.482
within-pixel, + temporal: 0.504


In [20]:
"""
Matched comparison: do front-loaded events flood differently from back-loaded events
that have similar rainfall and fall on cells in a similar state?
Assumes `ev` has the loading features (loading_index, nonpeak_share) from temporal_loading_split.py.
"""
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors

OUTCOME = "prop_floodable_area_flooded"
TREAT = "loading_index"                       # swap for any variable you want to isolate (e.g. antecedent moisture)
HI, LO = 0.3, -0.3                            # treated = front-loaded, control = back-loaded
MATCH_ON = ["max_precip", "total_acc", "length_of_active_rain", "hc_at_peak", "peak_cell_slope_avg", "fumax_at_peak",
           "mean_sm_2_before_event"]  # rainfall + cell state: edit
CALIPER = 1.0                                 # max allowed distance between matched events (in SD units of the matching variables)
N_BOOT = 1000

d = ev.dropna(subset=MATCH_ON + [TREAT, OUTCOME, "nonpeak_share"])
d = d[d["nonpeak_share"] > 0.4]               # only events where timing can matter
treated, control = d[d[TREAT] > HI], d[d[TREAT] < LO]
print(f"{len(treated)} treated, {len(control)} control events")

mu, sd = d[MATCH_ON].mean(), d[MATCH_ON].std()
Z = (d[MATCH_ON] - mu) / sd
nn = NearestNeighbors(n_neighbors=1).fit(Z.loc[control.index])
dist, pos = nn.kneighbors(Z.loc[treated.index])
keep = dist[:, 0] <= CALIPER                                   # drop treated events with no close control
t_idx = treated.index[keep]
c_idx = control.index[pos[keep, 0]]                            # matching with replacement
print(f"{keep.sum()} matched pairs ({keep.mean():.0%} of treated events within the caliper)")

# --- balance check: matching variables should look alike after matching (|SMD| < 0.1 is good) ---
def smd(a, b): return (a.mean() - b.mean()) / np.sqrt((a.var() + b.var()) / 2)
bal = pd.DataFrame({"SMD_before": {c: smd(treated[c], control[c]) for c in MATCH_ON},
                    "SMD_after": {c: smd(d.loc[t_idx, c], d.loc[c_idx, c]) for c in MATCH_ON}})
print("\nCovariate balance:\n", bal.round(2))

# --- effect: outcome difference within matched pairs ---
diff = d.loc[t_idx, OUTCOME].values - d.loc[c_idx, OUTCOME].values
rng = np.random.default_rng(0)
boot = [rng.choice(diff, len(diff)).mean() for _ in range(N_BOOT)]
print(f"\nMean outcome difference (front-loaded minus back-loaded): {diff.mean():.4f}, "
      f"95% CI [{np.percentile(boot, 2.5):.4f}, {np.percentile(boot, 97.5):.4f}]")
print(f"For scale: SD of {OUTCOME} among matched events = {d.loc[t_idx, OUTCOME].std():.4f}")

# --- does the effect depend on the cell? (e.g. by HC class: the regime question) ---
if "hc_class" in d:
    out = pd.DataFrame({"diff": diff, "hc_class": d.loc[t_idx, "hc_class"].values})
    print("\nEffect by HC class:\n", out.groupby("hc_class")["diff"].agg(["mean", "sem", "size"]).round(4))

# --- placebo: shuffle the treatment label among matched events; the effect should vanish ---
flip = rng.random(len(diff)) < 0.5
print("Placebo (random sign flips) mean difference:", round(np.where(flip, -diff, diff).mean(), 4))

1063 treated, 1396 control events
908 matched pairs (85% of treated events within the caliper)

Covariate balance:
                         SMD_before  SMD_after
max_precip                   -0.10       0.02
total_acc                     0.06       0.04
length_of_active_rain         0.14       0.02
hc_at_peak                   -0.09       0.01
peak_cell_slope_avg           0.04       0.03
fumax_at_peak                -0.06       0.01
mean_sm_2_before_event        0.02       0.00

Mean outcome difference (front-loaded minus back-loaded): 0.7812, 95% CI [0.6503, 0.9355]
For scale: SD of prop_floodable_area_flooded among matched events = 2.7625
Placebo (random sign flips) mean difference: -0.1492


0         3.035207
1         2.839798
2         3.460232
3         1.084400
4         3.169935
            ...   
106783    3.386704
106784    3.138564
106785    3.876193
106786    2.923936
106787    2.136064
Name: prop_floodable_area_flooded, Length: 105576, dtype: float64